# 💊 Drug Interaction Checker


## ── SECTION 0: Environment & Logging ──

In [1]:
import sys, platform, importlib, datetime, logging, os

# ─── Environment snapshot ────────────────────────────────
print("=" * 55)
print("ENVIRONMENT INFO")
print("=" * 55)
print(f"Date/Time : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python    : {sys.version}")
print(f"Platform  : {platform.platform()}")
for pkg in ["pandas", "difflib", "requests"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg:<18} {getattr(m, '__version__', 'stdlib')}")
    except ImportError:
        print(f"  {pkg:<18} NOT INSTALLED")
print("=" * 55)

# ─── Logging setup ───────────────────────────────────────
LOG_FILE = "drug_interaction_checker.log"
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger(__name__)
logger.info("Drug Interaction Checker started")
print(f"\nLogging → {LOG_FILE}")

ENVIRONMENT INFO
Date/Time : 2026-04-06 08:44:24
Python    : 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Platform  : Windows-10-10.0.26100-SP0
  pandas             2.3.3
  difflib            stdlib
  requests           2.32.5
2026-04-06 08:44:25,136 | INFO     | Drug Interaction Checker started

Logging → drug_interaction_checker.log


## DrugInteractionChecker Class ──

In [2]:
import pandas as pd
import difflib
from itertools import combinations
from typing import List, Dict, Optional, Tuple

# ─── Synonym dictionary (common name → canonical) ────────
DRUG_SYNONYMS: Dict[str, str] = {
    "acetaminophen":     "paracetamol",
    "tylenol":           "paracetamol",
    "panadol":           "paracetamol",
    "advil":             "ibuprofen",
    "motrin":            "ibuprofen",
    "nurofen":           "ibuprofen",
    "aspirin":           "acetylsalicylic acid",
    "asa":               "acetylsalicylic acid",
    "bayer":             "acetylsalicylic acid",
    "lasix":             "furosemide",
    "zoloft":            "sertraline",
    "prozac":            "fluoxetine",
    "plavix":            "clopidogrel",
    "coumadin":          "warfarin",
    "glucophage":        "metformin",
    "synthroid":         "levothyroxine",
    "lipitor":           "atorvastatin",
    "zocor":             "simvastatin",
    "nexium":            "esomeprazole",
    "prilosec":          "omeprazole",
    "xanax":             "alprazolam",
    "valium":            "diazepam",
    "amoxil":            "amoxicillin",
    "augmentin":         "amoxicillin-clavulanate",
    "zithromax":         "azithromycin",
    "norvasc":           "amlodipine",
    "lisinopril":        "lisinopril",
    "ventolin":          "albuterol",
    "aleve":             "naproxen",
    "naprosyn":          "naproxen",
}

# Comprehensive severity keyword sets
HIGH_KEYWORDS = frozenset([
    "severe", "avoid", "major", "contraindicated", "life-threatening",
    "fatal", "death", "serious", "do not", "do not use",
    "potentially fatal", "critically", "dangerous",
])
MEDIUM_KEYWORDS = frozenset([
    "moderate", "caution", "monitor", "increased risk",
    "may increase", "reduced", "decreased", "adjust",
])


class DrugInteractionChecker:
    """
    Checks drug–drug interactions using a CSV database.

    Expected CSV columns (case-insensitive):
        'Drug 1', 'Drug 2', 'Interaction Description'
    Optional column:
        'Severity'  (HIGH / MODERATE / LOW — used directly if present)
    """

    REQUIRED_COLS = {"drug 1", "drug 2", "interaction description"}

    def __init__(
        self,
        dataset_file: str = "db_drug_interactions.csv",
        fuzzy_threshold: float = 0.82,
        synonyms: Optional[Dict[str, str]] = None,
    ):
        self.fuzzy_threshold = fuzzy_threshold
        self.synonyms = {k.lower(): v.lower()
                         for k, v in (synonyms or DRUG_SYNONYMS).items()}
        self.db: Dict[str, Tuple[str, str]] = {}   # key → (severity, description)
        self.all_drug_names: List[str] = []
        self._load_database(dataset_file)

    # ── Database loading ─────────────────────────────────────
    def _load_database(self, filename: str) -> None:
        """Load CSV and build the interaction lookup map."""
        logger.info(f"Loading database from '{filename}' ...")
        try:
            df = pd.read_csv(filename, low_memory=False)
        except FileNotFoundError:
            logger.error(f"File not found: '{filename}'")
            raise FileNotFoundError(
                f"Database file '{filename}' not found. "
                "Place it in the working directory or supply the full path."
            )
        except Exception as e:
            logger.error(f"Failed to read CSV: {e}")
            raise

        # Normalise column names for validation
        df.columns = df.columns.str.strip()
        col_map = {c.lower(): c for c in df.columns}

        
        missing = self.REQUIRED_COLS - set(col_map.keys())
        if missing:
            available = list(df.columns)
            logger.error(f"Missing columns: {missing}. Available: {available}")
            raise ValueError(
                f"CSV is missing required columns: {missing}.\n"
                f"Available columns: {available}"
            )

        has_severity_col = "severity" in col_map

        
        d1_col   = col_map["drug 1"]
        d2_col   = col_map["drug 2"]
        desc_col = col_map["interaction description"]
        sev_col  = col_map.get("severity")

        # Drop rows with missing drug names
        df = df.dropna(subset=[d1_col, d2_col]).copy()
        df[d1_col]   = df[d1_col].astype(str).str.strip().str.lower()
        df[d2_col]   = df[d2_col].astype(str).str.strip().str.lower()
        df[desc_col] = df[desc_col].astype(str).str.strip()

        interaction_map: Dict[str, Tuple[str, str]] = {}
        for row in df[[d1_col, d2_col, desc_col]
                       + ([sev_col] if has_severity_col else [])].itertuples(index=False):
            d1, d2, desc = row[0], row[1], row[2]
            sev_raw = row[3] if has_severity_col else ""
            key = "_".join(sorted([d1, d2]))
            if key not in interaction_map:   # first occurrence wins
                severity = self._parse_severity(sev_raw, desc)
                interaction_map[key] = (severity, desc)

        self.db = interaction_map
        # Build sorted list of all drug names for fuzzy matching
        all_drugs: set = set()
        for key in self.db:
            parts = key.split("_", 1)
            all_drugs.update(parts)
        self.all_drug_names = sorted(all_drugs)

        logger.info(f"Database loaded: {len(self.db):,} unique interaction pairs")
        print(f"✅ Database ready — {len(self.db):,} interaction pairs loaded")

    # ── Severity parsing ─────────────────────────────────────
    @staticmethod
    def _parse_severity(sev_raw: str, description: str) -> str:
    
        sev_clean = sev_raw.strip().lower()
        # If CSV has a Severity column, trust it first
        if any(x in sev_clean for x in ["high", "major", "severe", "contraindicated"]):
            return "HIGH"
        if any(x in sev_clean for x in ["moderate", "medium"]):
            return "MEDIUM"
        if any(x in sev_clean for x in ["low", "minor", "minimal"]):
            return "LOW"

        # Fallback: scan description text
        desc_lower = description.lower()
        if any(kw in desc_lower for kw in HIGH_KEYWORDS):
            return "HIGH"
        if any(kw in desc_lower for kw in MEDIUM_KEYWORDS):
            return "MEDIUM"
        return "LOW"

    # ── Name normalisation ───────────────────────────────────
    def _normalise(self, name: str) -> str:
        """
        Normalise a drug name:
        1. Lowercase and strip.
        2. Resolve known synonyms.
        3. Optionally fuzzy-match to a known DB name.

        """
        clean = name.strip().lower()
        # Synonym lookup
        if clean in self.synonyms:
            logger.debug(f"Synonym: '{clean}' → '{self.synonyms[clean]}'")
            return self.synonyms[clean]
        # Exact match in DB
        if clean in set(self.all_drug_names):
            return clean
        # Fuzzy match
        if self.all_drug_names:
            matches = difflib.get_close_matches(
                clean, self.all_drug_names, n=1, cutoff=self.fuzzy_threshold
            )
            if matches:
                logger.debug(f"Fuzzy: '{clean}' → '{matches[0]}'")
                return matches[0]
        return clean

    # ── Core interaction check ───────────────────────────────
    def check_risk(
        self, drug_list: List[str]
    ) -> List[Dict[str, str]]:
        """
        Check all pairs from drug_list for known interactions.

        Returns a list of dicts: {Pair, Risk, Details, Drug1, Drug2}

        """
        if len(drug_list) < 2:
            logger.warning("check_risk called with fewer than 2 drugs.")
            return []

        normalised = [self._normalise(d) for d in drug_list]
        seen_keys: set = set()
        warnings: List[Dict] = []

        for d1, d2 in combinations(normalised, 2):
            key = "_".join(sorted([d1, d2]))
            if key in seen_keys:
                continue                
            seen_keys.add(key)

            if key in self.db:
                severity, desc = self.db[key]
                warnings.append({
                    "Pair":    f"{d1.title()} ⟷ {d2.title()}",
                    "Drug1":   d1,
                    "Drug2":   d2,
                    "Risk":    severity,
                    "Details": desc,
                })
                logger.info(f"Interaction found [{severity}]: {d1} + {d2}")

        # Sort: HIGH first, then MEDIUM, then LOW
        sev_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
        warnings.sort(key=lambda x: sev_order.get(x["Risk"], 9))
        return warnings

    # ── Convenience: summary report ──────────────────────────
    def report(self, drug_list: List[str]) -> str:
        """Return a formatted text report for the given drug list."""
        results = self.check_risk(drug_list)
        lines = [f"\n📋 Analyzing {len(drug_list)} drug(s):",
                 "   " + ", ".join(drug_list)]
        if not results:
            lines.append("\n✅ SAFE — No interactions found.")
        else:
            icons = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}
            lines.append(f"\n⚠️  FOUND {len(results)} INTERACTION(S):")
            for item in results:
                icon = icons.get(item["Risk"], "⚪")
                lines.append(f"  {icon} [{item['Risk']}]  {item['Pair']}")
                lines.append(f"      → {item['Details']}")
        return "\n".join(lines)


print("DrugInteractionChecker class defined.")

DrugInteractionChecker class defined.


## ── SECTION 2: Load Database ──

In [3]:
# Load once; reuse the checker instance throughout the session
try:
    checker = DrugInteractionChecker("db_drug_interactions.csv")
except FileNotFoundError as e:
    logger.error(str(e))
    print(f"\n❌ {e}")
    checker = None
except ValueError as e:
    logger.error(str(e))
    print(f"\n❌ CSV format error: {e}")
    checker = None

2026-04-06 08:45:30,442 | INFO     | Loading database from 'db_drug_interactions.csv' ...
2026-04-06 08:45:33,771 | INFO     | Database loaded: 191,135 unique interaction pairs
✅ Database ready — 191,135 interaction pairs loaded


## Evaluation / Test Suite ──

In [4]:
# Ground-truth test cases: (drugs_list, expected_interaction_exists, expected_severity_if_any)
# These are based on well-established pharmacology.
KNOWN_PAIRS = [
    # Positives (interactions that SHOULD be detected)
    (["warfarin",        "aspirin"],                     True,  "HIGH"),
    (["warfarin",        "acetylsalicylic acid"],         True,  "HIGH"),
    (["naproxen",        "ibuprofen"],                    True,  "MEDIUM"),
    (["brompheniramine", "dexbrompheniramine"],            True,  "MEDIUM"),
    (["clopidogrel",     "aspirin"],                      True,  None),   # severity may vary
    (["methotrexate",    "naproxen"],                     True,  None),
    (["lithium",         "ibuprofen"],                    True,  None),
    # Negatives (interactions that should NOT be detected)
    (["paracetamol",     "amoxicillin"],                  False, None),
    (["paracetamol",     "ibuprofen",     "aspirin"],      None,  None),   # at least one exists
    (["atorvastatin",    "lisinopril"],                   False, None),
]

def evaluate_checker(checker_instance, test_cases: list) -> dict:
    """Run test suite; compute precision/recall/accuracy."""
    if checker_instance is None:
        print("Checker not loaded — skipping evaluation.")
        return {}

    tp = fp = tn = fn = 0
    severity_correct = severity_total = 0

    print("\n" + "=" * 65)
    print("EVALUATION SUITE")
    print("=" * 65)
    print(f"{'Drugs':<45} {'Expect':>7} {'Got':>5} {'OK':>3}")
    print("-" * 65)

    for drugs, expected_interaction, expected_sev in test_cases:
        results   = checker_instance.check_risk(drugs)
        found_any = len(results) > 0
        label     = str(drugs)[:43]

        if expected_interaction is True:
            ok = "✓" if found_any else "✗"
            if found_any:
                tp += 1
            else:
                fn += 1
            # Check severity if specified
            if expected_sev and results:
                severity_total += 1
                if results[0]["Risk"] == expected_sev:
                    severity_correct += 1
        elif expected_interaction is False:
            ok = "✓" if not found_any else "✗"
            if not found_any:
                tn += 1
            else:
                fp += 1
        else:
            ok = "~"  # partial / don't-care

        got = "YES" if found_any else "NO"
        exp = "YES" if expected_interaction else ("NO" if expected_interaction is False else "?")
        print(f"  {label:<43} {exp:>7} {got:>5} {ok:>3}")

    print("-" * 65)
    total = tp + fp + tn + fn
    accuracy  = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp)   if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn)   if (tp + fn) > 0 else 0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0)
    sev_acc   = (severity_correct / severity_total
                 if severity_total > 0 else None)

    print(f"\nRESULTS (on {total} binary test cases):")
    print(f"  TP={tp}  FP={fp}  TN={tn}  FN={fn}")
    print(f"  Accuracy  : {accuracy:.2%}")
    print(f"  Precision : {precision:.2%}")
    print(f"  Recall    : {recall:.2%}")
    print(f"  F1 Score  : {f1:.2%}")
    if sev_acc is not None:
        print(f"  Severity  : {sev_acc:.2%} correct on {severity_total} labelled cases")
    print("=" * 65)

    metrics = dict(accuracy=accuracy, precision=precision,
                   recall=recall, f1=f1, tp=tp, fp=fp, tn=tn, fn=fn)
    logger.info(f"Evaluation: {metrics}")
    return metrics


if checker:
    eval_metrics = evaluate_checker(checker, KNOWN_PAIRS)


EVALUATION SUITE
Drugs                                          Expect   Got  OK
-----------------------------------------------------------------
2026-04-06 08:45:41,510 | INFO     | Interaction found [MEDIUM]: warfarin + acetylsalicylic acid
  ['warfarin', 'aspirin']                         YES   YES   ✓
2026-04-06 08:45:41,515 | INFO     | Interaction found [MEDIUM]: warfarin + acetylsalicylic acid
  ['warfarin', 'acetylsalicylic acid']            YES   YES   ✓
2026-04-06 08:45:41,518 | INFO     | Interaction found [LOW]: naproxen + ibuprofen
  ['naproxen', 'ibuprofen']                       YES   YES   ✓
2026-04-06 08:45:41,521 | INFO     | Interaction found [LOW]: brompheniramine + dexbrompheniramine
  ['brompheniramine', 'dexbrompheniramine']       YES   YES   ✓
2026-04-06 08:45:41,523 | INFO     | Interaction found [LOW]: clopidogrel + acetylsalicylic acid
  ['clopidogrel', 'aspirin']                      YES   YES   ✓
2026-04-06 08:45:41,528 | INFO     | Interaction found [LOW

## Programmatic API Usage ──

In [5]:
# ─── Example: use checker as a proper API ───────────────────
if checker:
    test_cases_api = [
        ["brompheniramine maleate", "oxymetazoline hydrochloride",
         "sodium ascorbate", "naproxen", "zinc sulfate monohydrate"],
        ["paracetamol", "ibuprofen", "aspirin"],
        ["naproxen", "ibuprofen", "aspirin"],
        [
            "Echinacea preparation", "anhydrous zinc acetate", "ascorbic acid",
            "brompheniramine", "brompheniramine maleate", "brompheniramine tannate",
            "dexbrompheniramine", "dexbrompheniramine maleate",
            "dexbrompheniramine tannate", "guaifenesin", "naproxen",
            "naproxen sodium", "oxymetazoline", "oxymetazoline hydrochloride",
            "phenindamine",
        ],
    ]

    for drug_list in test_cases_api:
        print(checker.report(drug_list))
        print()


📋 Analyzing 5 drug(s):
   brompheniramine maleate, oxymetazoline hydrochloride, sodium ascorbate, naproxen, zinc sulfate monohydrate

✅ SAFE — No interactions found.

2026-04-06 08:45:57,426 | INFO     | Interaction found [LOW]: propacetamol + ibuprofen
2026-04-06 08:45:57,428 | INFO     | Interaction found [LOW]: propacetamol + acetylsalicylic acid
2026-04-06 08:45:57,431 | INFO     | Interaction found [LOW]: ibuprofen + acetylsalicylic acid

📋 Analyzing 3 drug(s):
   paracetamol, ibuprofen, aspirin

⚠️  FOUND 3 INTERACTION(S):
  🟢 [LOW]  Propacetamol ⟷ Ibuprofen
      → The risk or severity of adverse effects can be increased when Ibuprofen is combined with Propacetamol.
  🟢 [LOW]  Propacetamol ⟷ Acetylsalicylic Acid
      → The risk or severity of adverse effects can be increased when Acetylsalicylic acid is combined with Propacetamol.
  🟢 [LOW]  Ibuprofen ⟷ Acetylsalicylic Acid
      → The risk or severity of adverse effects can be increased when Acetylsalicylic acid is combined w

## Interactive Session ──

In [8]:
def run_interactive(checker_instance: DrugInteractionChecker) -> None:
    """Interactive CLI for the drug interaction checker."""
    if checker_instance is None:
        print("Checker not loaded. Cannot run interactive session.")
        return

    print("\n" + "=" * 52)
    print(" DRUG INTERACTION CHECKER — Interactive Session")
    print("   Enter drugs separated by commas.")
    print("   Type 'exit' or 'quit' to stop.")
    print("=" * 52)

    while True:
        try:
            user_input = input("\n💊 Drugs: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if user_input.lower() in ("exit", "quit", ""):
            print("Goodbye! 👋")
            break

        drug_list = [d.strip() for d in user_input.split(",") if d.strip()]
        if len(drug_list) < 2:
            print("  ℹ️  Please enter at least 2 drugs.")
            continue

        print(checker_instance.report(drug_list))


# Uncomment the line below to start the interactive session:
run_interactive(checker)


 DRUG INTERACTION CHECKER — Interactive Session
   Enter drugs separated by commas.
   Type 'exit' or 'quit' to stop.



💊 Drugs:  Remifentanil, Fosphenytoin


2026-04-06 08:47:45,448 | INFO     | Interaction found [LOW]: remifentanil + fosphenytoin

📋 Analyzing 2 drug(s):
   Remifentanil, Fosphenytoin

⚠️  FOUND 1 INTERACTION(S):
  🟢 [LOW]  Remifentanil ⟷ Fosphenytoin
      → The risk or severity of adverse effects can be increased when Remifentanil is combined with Fosphenytoin.



💊 Drugs:  exit


Goodbye! 👋
